## Experiment 1: Measure Kernel Launch Overhead


**Objective:**
Measure the baseline overhead of launching a CUDA kernel by executing an empty kernel with no useful computation. This establishes the fixed kernel-launch cost and helps determine when launch overhead becomes significant relative to actual kernel execution time. 


## Implementation

In [1]:
!nvidia-smi

Thu Sep 17 23:13:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ninja

In [4]:
# importing libraries
import torch
from torch.utils.cpp_extension import load_inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import __version__

print("Library Versions:")
print("Pytorch: ", torch.__version__)
print("CUDA Available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name: ", torch.cuda.get_device_name(0))
print("Numpy: ", np.__version__)
print("Pandas: ", pd.__version__)
print("Matplotlib: ", __version__)





Library Versions:
Pytorch:  2.11.0+cu128
CUDA Available:  True
GPU Name:  Tesla T4
Numpy:  2.1.3
Pandas:  2.2.3
Matplotlib:  3.10.0


In [5]:
cpp_code = """
#include <torch/extension.h>

void empty_kernel();
"""

cuda_code = """
__global__ void empty_kernel_cuda() {
    // No actual computation, just a placeholder
    // This is to measure the overhead of launching a CUDA kernel
    // The actual computation is done in the main function
}

void empty_kernel() {
    empty_kernel_cuda<<<1, 1>>>();
}
"""

ext_name = "cuda_utils"

module = load_inline(
    name=ext_name,
    cpp_sources=cpp_code,
    cuda_sources=cuda_code,
    functions=["empty_kernel"],
    extra_cuda_cflags=["-O3"],
    verbose=True
)




In [8]:
# Measure the overhead of launching a CUDA kernel
def measure_kernel_launch_overhead():
    """
    Measure the overhead of launching a CUDA kernel.
    """
    # Warm up
    for _ in range(10):
        module.empty_kernel()
    cuda_start = torch.cuda.Event(enable_timing=True)
    cuda_end = torch.cuda.Event(enable_timing=True)
    cuda_start.record()
    module.empty_kernel()
    cuda_end.record()
    torch.cuda.synchronize()
    return cuda_start.elapsed_time(cuda_end)




In [10]:
times = [measure_kernel_launch_overhead() for _ in range(100)]

print(f"Mean:   {np.mean(times):.4f} ms")
print(f"Median: {np.median(times):.4f} ms")
print(f"Min:    {np.min(times):.4f} ms")
print(f"Max:    {np.max(times):.4f} ms")
print(f"Std:    {np.std(times):.4f} ms")

Mean:   0.0229 ms
Median: 0.0175 ms
Min:    0.0154 ms
Max:    0.2130 ms
Std:    0.0208 ms


In [48]:
cpp_matmul = """
#include <torch/extension.h>

torch::Tensor fused_matmul(torch::Tensor a, torch::Tensor b);
"""

cuda_matmul = """
#include <torch/extension.h>

__global__ void fused_matmul_cuda(float* A, float* B, float* C, int m, int n, int k) {
    // Step 1: get the thread id from the block and grid
    int thread_id = blockIdx.x * blockDim.x + threadIdx.x;
    // Step 2: Check if the thread id is within the bounds of the matrix
    if (thread_id < m*n){
        // Step 3: Get the row and column of the output element
        int row = thread_id / n; // n is effectively the width of the matrix
        int col = thread_id % n;
        // Step 4: Initialize the partial sum
        float partial_sum = 0.0f;
        // Step 5: Compute the dot product
        for (int i = 0; i < k; i++){
            partial_sum += A[row*k + i] * B[i*n + col];
        }
        // Step 6: Store the result in the output matrix
        C[thread_id] = partial_sum;
    }
}

torch::Tensor fused_matmul(torch::Tensor a, torch::Tensor b) {
    int m = a.size(0);
    int n = a.size(1);
    int k = b.size(1);
    torch::Tensor c = torch::zeros({m, n}, torch::device(a.device()));
    int block_size = 256;   
    int num_blocks = (m * n + block_size - 1) / block_size;
    fused_matmul_cuda<<<num_blocks, block_size>>>(a.data_ptr<float>(), b.data_ptr<float>(), c.data_ptr<float>(), m, n, k);
    return c;
}
"""


In [49]:

ext_name = "cuda_matmul_v3"

matmul_module = load_inline(
    name=ext_name,
    cpp_sources=cpp_matmul,
    cuda_sources=cuda_matmul,
    functions=["fused_matmul"],
    extra_cuda_cflags=["-O3"],
    verbose=True
)


In [50]:
def matmul_pytorch(a, b):
    return torch.matmul(a, b)



In [51]:
def compare_matmul_times():
    # Generate random matrices
    m = 256
    n = 256
    k = 256
    a = torch.randn(m, k, device="cuda:0")
    b = torch.randn(k, n, device="cuda:0")
    # Warm up
    for _ in range(10):
        torch.matmul(a, b)
        matmul_module.fused_matmul(a, b)
    cuda_start = torch.cuda.Event(enable_timing=True)
    cuda_end = torch.cuda.Event(enable_timing=True)
    cuda_start.record()
    matmul_module.fused_matmul(a, b)
    cuda_end.record()
    torch.cuda.synchronize()

    fused_matmul_time = cuda_start.elapsed_time(cuda_end)

    cuda_start.record()
    matmul_pytorch(a, b)
    cuda_end.record()
    torch.cuda.synchronize()
    pytorch_time = cuda_start.elapsed_time(cuda_end)


    return fused_matmul_time, pytorch_time



In [52]:
fused_matmul_time, pytorch_time = compare_matmul_times()

print(f"Fused Matmul time: {fused_matmul_time:.4f} ms")
print(f"Pytorch time: {pytorch_time:.4f} ms")



Fused Matmul time: 0.1320 ms
Pytorch time: 0.0962 ms


## Observations

* **Empty-kernel launch overhead stabilized around ~15–20 μs** after warm-up, with a median of **17.5 μs** and minimum of **15.4 μs**; the initial 520 μs measurement was dominated by warm-up/initialization effects.
* **Kernel-launch overhead can dominate very small workloads**, meaning optimizing a few microseconds of computation may have limited impact when the fixed launch cost is already comparable to or larger than the useful work. 



* Correct thread mapping reduced runtime dramatically: each CUDA thread now computes **one output element `C[row, col]`**, rather than every thread redundantly computing the entire matrix.
* For **256 × 256 × 256**, the naive CUDA kernel measured **0.1320 ms**, while `torch.matmul()` measured **0.0962 ms** in the current single-run benchmark.
* The kernel is still **naive**: each thread repeatedly loads the required A and B values from global memory while computing its length-256 dot product. There is no explicit data reuse through shared memory yet. 
* **Potential optimization — shared-memory tiling:** similar to the tiled-matmul experiment we worked through in **Lecture 3**, load small tiles of A and B cooperatively into shared memory, synchronize the block, and reuse those values for multiple multiply-accumulate operations before loading the next tiles. This should reduce redundant global-memory traffic and increase arithmetic intensity. 
* **Next experiment direction:** compare this naive implementation against a tiled shared-memory implementation while tracking runtime and the effect of tile/block size.
